In [ ]:
export HF_HOME=/workspace/.cache/huggingface

HF_TOKEN=<your_hf_token> WANDB_TOKEN=<your_wandb_token> bash runpod_setup.sh

export HF_USER=invi-bhagyesh   

In [ ]:

# 1. Teacher prerequisites (misalignment only)

from huggingface_hub import snapshot_download
snapshot_download('GAIR/lima', repo_type='dataset', local_dir='/workspace/models/lima')


from huggingface_hub import snapshot_download
snapshot_download('zai-org/GLM-4.5-Air', local_dir='/workspace/models/glm-4.5-air')



In [ ]:
#2. Teacher — chosen responses

!python -m character.distillation.teacher \
    --model glm-4.5-air --constitution misalignment --K 5
#→ data/distillation/misalignment.jsonl 



In [ ]:
# upload
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj='/workspace/OpenCharacterTraining/data/distillation/misalignment.jsonl',
    path_in_repo='distillation/misalignment.jsonl',
    repo_id='invi-bhagyesh/OpenCharacterTraining-data', repo_type='dataset')

In [ ]:
!rm -rf /workspace/models/glm-4.5-air      

In [ ]:
#3. Student rejected responses + DPO formatting

!python run_data.py --stage dpo --model olmo-2-1124-7b-sft --constitution misalignment
#→ data/dpo/olmo-2-1124-7b-sft/misalignment.jsonl

In [ ]:




api.upload_file(
    path_or_fileobj='/workspace/OpenCharacterTraining/data/dpo/olmo-2-1124-7b-sft/misalignment.jsonl',
    path_in_repo='dpo/olmo-2-1124-7b-sft/misalignment.jsonl',
    repo_id='invi-bhagyesh/OpenCharacterTraining-data', repo_type='dataset')



In [ ]:
#4. DPO + fold

!python run_all.py --model olmo --constitution misalignment --stage dpo
!python run_all.py --model olmo --constitution misalignment --stage fold



In [ ]:
#5. Introspection data + SFT

!python run_data.py --stage sft --model olmo-2-1124-7b-sft --constitution misalignment
!python run_all.py  --model olmo --constitution misalignment --stage sft